In [10]:
# CELL 1 [Markdown]
"""
# 🛒 Google Ads x WooCommerce Shopping Matching Pipeline

Ce notebook permet d'analyser les termes de recherche Google Ads et de les croiser (*matching*) avec le flux de produits WooCommerce.

### Objectifs :
1. **Identifier les opportunités SEO/Shopping** : Mots-clés générateurs de conversions absents des titres de produits WooCommerce.
2. **Identifier le gaspillage** : Mots-clés coûteux sans aucune conversion pour alimenter la liste de mots-clés négatifs.
3. **Préparer le pipeline** pour l'automatisation via GitHub Actions.
"""

# CELL 2 [Code] : Ingestion & Chargement des Données
import pandas as pd
import requests
import gdown

# 1. Configuration des IDs Google Drive
FILE_ID_ADS = '1_8quOdA863-70Q-vOfIRrvC6qe9eGAht'
FILE_ID_FLUX = '1GAO2lShlKXF0kjG8qdtevKlqnkSt3W8z'

# 2. Téléchargement sécurisé
print("Téléchargement des fichiers en cours...")
gdown.download(id=FILE_ID_ADS, output='ads_data.xlsx', quiet=True)
gdown.download(id=FILE_ID_FLUX, output='flux_shopping.xml', quiet=True)

# 3. Chargement Google Ads (.xlsx)
print("\n Chargement des données Google Ads...")
df_ads = pd.read_excel('ads_data.xlsx', header=2)
df_ads.columns = df_ads.columns.str.strip()
print(f"Google Ads chargé avec succès ({len(df_ads)} lignes).")
display(df_ads.head(3))

# 4. Nettoyage XML et Chargement du Flux WooCommerce
print("\n Nettoyage et chargement du flux produits WooCommerce...")

with open('flux_shopping.xml', 'rb') as f:
    content = f.read().replace(b'\x00', b'')

with open('flux_shopping_clean.xml', 'wb') as f:
    f.write(content)

try:
    df_flux = pd.read_xml('flux_shopping_clean.xml', xpath='.//item')
    print(f"Flux XML chargé avec succès ({len(df_flux)} produits) !")
    display(df_flux.head(3))
except Exception as e:
    print(f"Tentative xpath standard... ({e})")
    df_flux = pd.read_xml('flux_shopping_clean.xml')
    display(df_flux.head(3))

# CELL 3 [Markdown]
"""
## Étape 2 : Traitement des Métriques, Matching & Export
"""

# CELL 4 [Code] : Matching, Métriques & Exports CSV
import numpy as np

# 1. Filtrage et calcul des métriques Ads
print("Traitement des données Google Ads...")

# 1. NETTOYAGE STRICT DES LIGNES DE TOTAL / RÉSUMÉ GOOGLE ADS
# On supprime toutes les lignes commençant par "Total" ou contenant "Total :"
df_ads = df_ads[df_ads['Terme de recherche'].notna()].copy()
df_ads['Terme de recherche'] = df_ads['Terme de recherche'].astype(str).str.strip()

df_ads = df_ads[~df_ads['Terme de recherche'].str.startswith('Total')].copy()
df_ads = df_ads[~df_ads['Terme de recherche'].str.contains('Total :', case=False, na=False)].copy()

# 2. CONVERSION NUMÉRIQUE
df_ads['Conversions'] = pd.to_numeric(df_ads['Conversions'], errors='coerce').fillna(0)
df_ads['Coût'] = pd.to_numeric(df_ads['Coût'], errors='coerce').fillna(0)

# 3. FILTRAGE ET CALCUL DES MÉTRIQUES
termes_convertisseurs = df_ads[df_ads['Conversions'] > 0].copy()
termes_convertisseurs['CPA'] = termes_convertisseurs['Coût'] / termes_convertisseurs['Conversions']

if 'Valeur de conv./coût' in termes_convertisseurs.columns:
    termes_convertisseurs['ROAS'] = pd.to_numeric(termes_convertisseurs['Valeur de conv./coût'], errors='coerce')
else:
    termes_convertisseurs['ROAS'] = 0

print(f"Mots-clés convertisseurs identifiés (sans les totaux) : {len(termes_convertisseurs)}")

# 4. MATCHING AVEC LES TITRES WOOCOMMERCE
print("\n Recherche de correspondance avec le flux WooCommerce...")

titres_flux = df_flux['title'].astype(str).str.lower().tolist()

def terme_est_dans_flux(terme):
    terme_clean = str(terme).lower().strip()
    return any(terme_clean in titre for titre in titres_flux)

termes_convertisseurs['Present_dans_WooCommerce'] = termes_convertisseurs['Terme de recherche'].apply(terme_est_dans_flux)

# 5. EXPORTS & RÉSULTATS
opportunites = termes_convertisseurs[termes_convertisseurs['Present_dans_WooCommerce'] == False].sort_values(
    by='Conversions', ascending=False
)

mots_cles_negatifs = df_ads[(df_ads['Conversions'] == 0) & (df_ads['Coût'] > 10)].sort_values(
    by='Coût', ascending=False
)

print(f"\n OPPORTUNITÉS TROUVÉES : {len(opportunites)} mots-clés rentables absents de vos titres WooCommerce !")
print(f"MOTS-CLÉS NÉGATIFS POTENTIELS : {len(mots_cles_negatifs)} termes à fort coût sans conversion.")

print("\n--- TOP 10 DES MOTS-CLÉS À AJOUTER DANS VOS TITRES WOOCOMMERCE ---")
display(opportunites[['Terme de recherche', 'Conversions', 'Coût', 'CPA', 'ROAS']].head(10))

opportunites.to_csv('opportunites_titres_woocommerce.csv', index=False)
mots_cles_negatifs.to_csv('mots_cles_negatifs_shopping.csv', index=False)
print("\n Fichiers CSV d'analyse générés avec succès !")

Téléchargement des fichiers en cours...

 Chargement des données Google Ads...
Google Ads chargé avec succès (32918 lignes).


,Terme de recherche,Type de correspondance,Ajoutée/Exclue,Campagne,Groupe d'annonces,Clics,Impr.,CTR,Code de la devise,CPC moy.,Coût,Valeur de conv./coût,Taux de conv.,Conversions,Coût/conv.
0,ceinture karaté adulte,Requête large,Aucun,[S] - Karate-gi-KARATE,Ceintures-Karate,1,1,1,EUR,0.12,0.12,3725.00,2.0000,2.00,0.06
1,kimono kamikaze emperor,Performance Max,Aucun,Karate-Gi-Performance Max-4,--,1,26,0.0385,EUR,0.12,0.12,3447.50,1.0000,1.00,0.12
2,karate g,Mot clé exact (variante proche),Ajouté,[S] - Karate-gi-KARATE,Kimonos-Karate,7,13,0.5385,EUR,0.06,0.45,2180.01,0.5274,3.69,0.12



 Nettoyage et chargement du flux produits WooCommerce...
Flux XML chargé avec succès (7574 produits) !


,id,title,description,link,image_link,additional_image_link,availability,price,gender,age_group,identifier_exists,mpn,brand,custom_label_4,item_group_id,size,color,custom_label_0,material
0,3334205,Kimono Karate Arawaza Rev-X réversible WUKF,Kimono Karate Arawaza Rev-X réversible WUKF Ki...,https://karate-gi.fr/kimono-karate-arawaza-rev...,https://karate-gi.fr/wp-content/uploads/Kimono...,https://karate-gi.fr/wp-content/uploads/Kimono...,backorder,276.98 EUR,unisex,adult,yes,3143590-143593,Arawaza,Sur Commande,NaN,None,None,None,None
1,3334369,"En stock, Habituellement expédié sous 10 à 20 ...",Kimono Karate Arawaza Rev-X réversible WUKF Ki...,https://karate-gi.fr/kimono-karate-arawaza-rev...,https://karate-gi.fr/wp-content/uploads/Kimono...,https://karate-gi.fr/wp-content/uploads/Kimono...,backorder,276.98 EUR,unisex,adult,yes,1510-6,Arawaza,Sur Commande,3334205.0,150,None,None,None
2,3334215,"En stock, Habituellement expédié sous 10 à 20 ...",Kimono Karate Arawaza Rev-X réversible WUKF Ki...,https://karate-gi.fr/kimono-karate-arawaza-rev...,https://karate-gi.fr/wp-content/uploads/Kimono...,https://karate-gi.fr/wp-content/uploads/Kimono...,backorder,339.90 EUR,unisex,adult,yes,1510-14,Arawaza,Sur Commande,3334205.0,195,None,None,None


Traitement des données Google Ads...
Mots-clés convertisseurs identifiés (sans les totaux) : 340

 Recherche de correspondance avec le flux WooCommerce...

 OPPORTUNITÉS TROUVÉES : 253 mots-clés rentables absents de vos titres WooCommerce !
MOTS-CLÉS NÉGATIFS POTENTIELS : 19 termes à fort coût sans conversion.

--- TOP 10 DES MOTS-CLÉS À AJOUTER DANS VOS TITRES WOOCOMMERCE ---


,Terme de recherche,Conversions,Coût,CPA,ROAS
121,karategi,158.58,1249.07,7.876592,14.55
103,karaté gi,80.53,603.77,7.497454,18.68
138,karategi,63.14,278.96,4.418118,10.32
91,karate gi fr,49.61,189.32,3.816166,21.04
59,karategi fr,41.89,98.13,2.342564,39.52
136,karaté gi,36.45,147.43,4.044719,10.43
157,karategi,20.76,206.93,9.967726,7.05
150,karategi,13.70,176.50,12.883212,8.25
64,karate gi fr,13.60,45.16,3.320588,36.00
139,karategi tokaido,12.01,68.73,5.722731,9.88



 Fichiers CSV d'analyse générés avec succès !
